In [1]:
from datasets import DatasetDict,Dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer
import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

In [2]:
from datasets import load_dataset

ds = load_dataset("shawhin/phishing-site-classification")

In [3]:
model_name = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
id2label={
    0:"Safe",
    1:"Not safe"
}
label2id={
    "Safe":0,
    "Not safe":1
}
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# Freezing all base model parameters
for name,param in model.base_model.named_parameters():
    param.requires_grad=False

for name,param in model.base_model.named_parameters():
    if "pooler" in name:
        param.requires_grad=True

In [7]:
#Processing of data
def process_function(examples):
    return tokenizer(examples["text"],truncaton=True)

tokenized_data=DatasetDict.map(process_function,batched=True)

AttributeError: 'function' object has no attribute '_check_values_type'